# Vollständiger App-Durchlauf im Jupyter-Notebook

Dieses Notebook ist eine **Lern- und Testumgebung für den Dialogkern der SOB-App**. Du durchläufst den gleichen fachlichen Ablauf wie in der App und siehst zusätzlich den internen Zustand.

**So arbeitest du damit**

1. Führe die Zellen beim ersten Durchlauf von oben nach unten mit **Shift + Enter** aus.
2. Ändere nur Variablen, die mit **„HIER ÄNDERN“** markiert sind.
3. Führe eine Zelle mit **„nur einmal“** während desselben Gesprächs nicht erneut aus.
4. Beginne für einen neuen Test wieder bei Schritt 4.
5. Verwende ausschließlich synthetische Testdaten.

> Das Notebook prüft die Backend-Logik. Darstellung, Schaltflächen und Bedienung im Browser müssen abschließend in der echten App geprüft werden.


## 1 – Was passiert in der App?

```mermaid
flowchart TD
    A[CSV und optional alter SOB] --> B[Import]
    B --> C[Dokumentation Lebensfeldern zuordnen]
    C --> D[Lebensfelder fachlich prüfen]
    D --> E[Einteilung bestätigen]
    E --> F[Ziele und Beobachtungen je Lebensfeld besprechen]
    F --> G[Berichtsentwurf erstellen und prüfen]
    G --> H[Lebensfeld freigeben]
    H --> I{Weitere aktive Felder?}
    I -- Ja --> F
    I -- Nein --> J[DOCX oder Markdown exportieren]
```

Der zentrale Gesprächszustand heißt `ChatSession`. Darin stehen die aktuelle Phase, das aktive Lebensfeld, der Nachrichtenverlauf, gespeicherte Beobachtungen, Unsicherheiten, Ziele und Entwürfe.


## 2 – Notebook vorbereiten

**Wann?** Einmal nach dem Öffnen oder nach einem Kernel-Neustart.

**Was passiert?** Das Projekt wird gefunden, benötigte Klassen werden importiert und Hilfsanzeigen werden definiert. Es wird noch kein Modell geladen und keine Sitzung gestartet.


In [ ]:
from pathlib import Path
import sys
from html import escape
from IPython.display import HTML, Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'sob_generator').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'sob_generator').exists():
    raise RuntimeError('Projektordner sob-generator-lab wurde nicht gefunden.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sob_generator.chat_engine import ChatDialogueEngine
from sob_generator.chat_store import ChatSessionStore
from evals.walkthrough_log import write_walkthrough_log

PHASEN = {
    'screening': 'Auswahl der Lebensfelder prüfen',
    'field_reflect': 'Ziele und vorhandene Informationen prüfen',
    'field_goals': 'Ziele bearbeiten',
    'field_questions': 'Fehlende Angaben klären',
    'field_draft': 'Entwurf liegt vor',
    'field_review': 'Entwurf fachlich prüfen',
    'finalized': 'Alle ausgewählten Lebensfelder freigegeben',
}


def zeige_nachrichten(nachrichten):
    for nachricht in nachrichten:
        rolle = 'Fachkraft' if nachricht.role.value == 'user' else 'Chatbot'
        display(Markdown(f'**{rolle}:**\n\n{nachricht.content}'))


def zeige_status(sitzung):
    aktives_feld = sitzung.sections.get(sitzung.active_section_id)
    aktive = [zustand for zustand in sitzung.sections.values() if zustand.is_active_field]
    bestaetigte = [zustand for zustand in sitzung.sections.values() if zustand.status.value == 'confirmed']
    display(HTML(
        '<table><tr><th>Merkmal</th><th>Aktueller Wert</th></tr>'
        f'<tr><td>Phase</td><td>{escape(PHASEN.get(sitzung.phase.value, sitzung.phase.value))}</td></tr>'
        f'<tr><td>Aktives Lebensfeld</td><td>{escape(aktives_feld.section_id + " – " + aktives_feld.title) if aktives_feld else "–"}</td></tr>'
        f'<tr><td>Ausgewählte Lebensfelder</td><td>{len(aktive)}</td></tr>'
        f'<tr><td>Freigegebene Lebensfelder</td><td>{len(bestaetigte)} von {len(sitzung.sections)}</td></tr>'
        f'<tr><td>Nachrichten</td><td>{len(sitzung.messages)}</td></tr></table>'
    ))

PROTOKOLL_ORDNER = REPO_ROOT / 'output/walkthrough'
print(f'Projekt gefunden: {REPO_ROOT}')
print(f'Automatisches Protokoll: {PROTOKOLL_ORDNER}')


## 3 – Modell laden

**Wann?** Einmal pro Kernel-Sitzung.

- `ECHTES_MODELL = True`: echte fachliche Prüfung mit dem lokalen 7B-Modell.
- `ECHTES_MODELL = False`: schneller technischer Probelauf; nicht zur Bewertung der Antwortqualität geeignet.

Das Laden kann beim ersten Mal dauern. Danach wird dasselbe Modell für das gesamte Gespräch weiterverwendet.


In [ ]:
ECHTES_MODELL = True  # HIER ÄNDERN: Für fachliche Tests auf True lassen.

if ECHTES_MODELL:
    from sob_generator.llm.generator import LLMDraftGenerator
    modellpfad = REPO_ROOT / 'models/Qwen2.5-7B-Instruct-Q4_K_M.gguf'
    if not modellpfad.exists():
        raise FileNotFoundError(f'Modell nicht gefunden: {modellpfad}')
    generator = LLMDraftGenerator(str(modellpfad), verbose=False)
    print('Echtes 7B-Modell ist bereit.')
else:
    from sob_generator.stub_generator import StubDraftGenerator
    generator = StubDraftGenerator()
    print('Technischer Stub ist bereit. Antworten nicht fachlich bewerten.')


## 4 – Synthetische Testdateien festlegen

**Wann?** Einmal vor dem Start eines neuen Gesprächs.

**Was passiert?** Hier werden nur die Dateipfade festgelegt und geprüft. Die Dateien werden erst in Schritt 5 importiert. Der CSV-Export enthält Verlaufsdokumentation; der alte SOB liefert frühere Ziele.


In [ ]:
CSV_DATEI = REPO_ROOT / 'dummy/Max_Mustermann_Vivendi_Export.csv'
ALTER_SOB = REPO_ROOT / 'dummy/Max_Mustermann_SVB_ausgefuellt.docx'

for datei in (CSV_DATEI, ALTER_SOB):
    if not datei.exists():
        raise FileNotFoundError(f'Testdatei fehlt: {datei}')

print(f'CSV: {CSV_DATEI.name}')
print(f'Alter SOB: {ALTER_SOB.name}')
print('Beide synthetischen Testdateien sind vorhanden.')


## 5 – Echte App-Sitzung starten

**Wann?** Genau einmal pro Testgespräch. Erneutes Ausführen löscht den bisherigen Gesprächsstand und startet von vorn.

**Was passiert intern?**

1. CSV und alter SOB werden eingelesen.
2. Das Modell ordnet Dokumentation den Lebensfeldern zu.
3. Frühere Ziele werden übernommen.
4. Eine neue `ChatSession` wird im Arbeitsspeicher angelegt.
5. Der Chatbot erzeugt Begrüßung und Vorschlag zur Auswahl der Lebensfelder.

Das entspricht fachlich dem Upload und Start der interaktiven App.


In [ ]:
chat_store = ChatSessionStore()
chat_engine = ChatDialogueEngine(generator, session_store=chat_store)

chat_session = await chat_engine.start_session(
    csv_path=CSV_DATEI,
    alt_sob_path=ALTER_SOB,
    anrede='Herr',
)

print('App-Sitzung gestartet. Begrüßung und Einteilungsvorschlag:')
zeige_nachrichten(chat_session.messages)
zeige_status(chat_session)
protokoll_pfad, status_pfad = write_walkthrough_log(chat_engine, chat_session, PROTOKOLL_ORDNER)
print(f'Protokoll gestartet: {protokoll_pfad}')


## 6 – Vorgeschlagene Lebensfelder prüfen

Die Sitzung befindet sich zunächst in der Phase **Screening**. Prüfe die Begründungen und die vorausgewählten Lebensfelder. In der Browser-App würdest du dieselben Informationen in der Lebensfeldübersicht sehen.

Du kannst im nächsten Dialogschritt beispielsweise schreiben:

- `Warum wurde Lebensfeld 1.1 ausgewählt?`
- `Feld 2.3 zusätzlich aktivieren.`
- `Feld 1.2 abwählen.`
- `Einteilung passt.`

Erst `Einteilung passt.` beendet das Screening und öffnet das erste aktive Lebensfeld. Diese Bestätigung bestätigt nur die **Auswahl**, noch keine Inhalte.


In [ ]:
rows = []
for feld_id, zustand in chat_session.sections.items():
    auswahl = 'ja' if zustand.is_active_field else 'nein'
    rows.append(
        f'<tr><td>{escape(feld_id)}</td><td>{escape(zustand.title)}</td>'
        f'<td>{auswahl}</td><td>{escape(zustand.screening_rationale or "–")}</td></tr>'
    )
display(HTML(
    '<table><thead><tr><th>ID</th><th>Lebensfeld</th><th>Ausgewählt</th><th>Begründung</th></tr></thead>'
    '<tbody>' + ''.join(rows) + '</tbody></table>'
))


## 7 – Fortlaufenden Fachkraftdialog führen

Diese Zelle ist dein Eingabefeld für **alle Gesprächsphasen**.

1. Ändere nur `NACHRICHT`.
2. Führe die Zelle mit **Shift + Enter** aus.
3. Lies Antwort und Phase.
4. Ändere `NACHRICHT` und führe dieselbe Zelle erneut aus.

Solange du Schritt 5 nicht erneut ausführst, bleibt der gesamte Dialog erhalten.


In [ ]:
async def sende_fachkraftnachricht(nachricht):
    if not nachricht.strip():
        raise ValueError('Bitte eine Nachricht eintragen.')
    anzahl_vorher = len(chat_session.messages)
    aktuelle_sitzung = await chat_engine.process_message(chat_session.session_id, nachricht)
    neue_nachrichten = aktuelle_sitzung.messages[anzahl_vorher:]
    zeige_nachrichten(neue_nachrichten)
    zeige_status(aktuelle_sitzung)
    protokoll_pfad, status_pfad = write_walkthrough_log(
        chat_engine, aktuelle_sitzung, PROTOKOLL_ORDNER
    )
    print(f'Protokoll aktualisiert: {protokoll_pfad.name}')
    return aktuelle_sitzung

NACHRICHT = 'Einteilung passt.'  # HIER ÄNDERN: erste Nachricht der Fachkraft
chat_session = await sende_fachkraftnachricht(NACHRICHT)


### Beispiel: vollständiger Einkaufsdialog mit mehreren Fachkraftantworten

Führe die folgenden Zellen **einzeln und in dieser Reihenfolge** aus. Lies nach jeder Zelle zuerst die Antwort des Chatbots. Die Beispielantworten zeigen einen realistischen Verlauf, sind aber keine fachlich vorgeschriebenen Antworten. Wenn die Rückfrage des Chatbots nicht zur vorbereiteten nächsten Antwort passt, ändere deren `NACHRICHT` passend ab.

Der Dialog wechselt bewusst zu **Lebensfeld 1.2 – Alltägliche Lebensführung / Hauswirtschaft**, weil Einkaufen dort eingeordnet ist.


**Fachkraftantwort 2 – passendes Lebensfeld öffnen**


In [ ]:
NACHRICHT = 'Wechsel zu Feld 1.2.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 3 – aktuelles Ziel benennen**


In [ ]:
NACHRICHT = 'Das aktuelle Ziel ist, dass Herr Mustermann seine Einkäufe zunehmend selbstständig plant und erledigt.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 4 – konkrete Beobachtung ergänzen**


In [ ]:
NACHRICHT = 'Bei den letzten vier Einkäufen schrieb Herr Mustermann die Einkaufsliste zweimal selbst und fand die benötigten Waren ohne Hinweis.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 5 – benötigte Unterstützung beschreiben**


In [ ]:
NACHRICHT = 'Die Assistenz erinnerte ihn vor dem Einkauf an die Liste und unterstützte ihn einmal beim Bezahlen mit der Bankkarte.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 6 – Wunsch und Zeitraum konkretisieren**


In [ ]:
NACHRICHT = 'Herr Mustermann möchte bis zum 31.12.2026 drei von vier geplanten Wocheneinkäufen mit seiner Liste selbstständig erledigen.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 7 – eine Unsicherheit offen benennen**


In [ ]:
NACHRICHT = 'Ob er das Bezahlen mit der Bankkarte zuverlässig allein bewältigt, kann ich noch nicht beurteilen.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 8 – eine frühere Angabe korrigieren**


In [ ]:
NACHRICHT = 'Korrektur: Bei den letzten vier Einkäufen schrieb Herr Mustermann die Einkaufsliste nur einmal selbst.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 9 – Zielvorschlag anfordern**


In [ ]:
NACHRICHT = 'Bitte formuliere aus den vorhandenen Angaben einen überprüfbaren Zielvorschlag und lasse die offene Frage zum Bezahlen sichtbar.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


**Fachkraftantwort 10 – Berichtsentwurf anfordern**


In [ ]:
NACHRICHT = 'Bitte erstelle jetzt einen bearbeitbaren Entwurf für dieses Lebensfeld.'
chat_session = await sende_fachkraftnachricht(NACHRICHT)


Führe danach **Schritt 8 – Internen Zustand prüfen** aus. Kontrolliere besonders:

- Wurde die Beobachtung „zweimal selbst“ durch „nur einmal selbst“ ersetzt?
- Bleibt die Unsicherheit zum Bezahlen sichtbar?
- Enthält das Ziel nur belegte Angaben?
- Ist der Entwurf klar von Beobachtung, fachlicher Einordnung und Zielvorschlag getrennt?


### Sinnvolle Simulation eines Fachkraftdialogs

Führe Schritt 7 nacheinander mit frei formulierten Varianten dieser Inhalte aus:

1. **Screening:** Auswahl hinterfragen, verändern und schließlich bestätigen.
2. **Beobachtung:** Eine konkrete beobachtete Situation beschreiben.
3. **Unterstützung:** Art und Umfang der Hilfe ergänzen.
4. **Wunsch:** Den Wunsch der betreuten Person benennen.
5. **Zeitraum/Messbarkeit:** Fehlende Zielmerkmale nachreichen.
6. **Unsicherheit:** Offen sagen, was noch nicht beurteilt werden kann.
7. **Korrektur:** Eine vorherige Angabe mit `Korrektur:` berichtigen.
8. **Entwurf:** `Entwurf erstellen.` schreiben.
9. **Revision:** Eine konkrete Änderung am Entwurf verlangen.

Bewerte nach jeder Antwort: Quellenbindung, fachliche Angemessenheit, Verständlichkeit der nächsten Handlung und korrekte Speicherung.


## 8 – Internen Zustand prüfen

**Wann?** Nach jeder fachlich wichtigen Nachricht.

Die sichtbare Antwort kann plausibel sein, obwohl intern etwas Falsches gespeichert wurde. Diese Kontrolle zeigt deshalb Beobachtungen, Unsicherheiten, Ziele, offenen Klärungsbedarf und den aktuellen Berichtsentwurf des aktiven Lebensfelds.


In [ ]:
zustand = chat_session.sections[chat_session.active_section_id]

print(f'Aktives Lebensfeld: {zustand.section_id} – {zustand.title}')
print(f'Phase: {PHASEN.get(chat_session.phase.value, chat_session.phase.value)}')
print(f'Status: {zustand.status.value}')

print('\nGespeicherte Beobachtungen:')
for wert in zustand.user_notes:
    print(f'- {wert}')
if not zustand.user_notes:
    print('– keine –')

print('\nUnsicherheiten:')
for wert in zustand.uncertainties:
    print(f'- {wert}')
if not zustand.uncertainties:
    print('– keine –')

print('\nAktuelle Ziele:')
print(zustand.current_goals or '– keine –')
print('\nOffene Rückfrage:')
print(zustand.pending_question or '– keine –')
print('\nBerichtsentwurf:')
print(zustand.bericht_zielerreichung or '– noch keiner –')


## 9 – Aktives Lebensfeld wechseln

**Wann?** Wenn du gezielt ein anderes ausgewähltes Lebensfeld testen möchtest.

In der Browser-App entspricht dies dem Klick auf ein Lebensfeld. Trage eine ID aus der Tabelle in Schritt 6 ein. Der jeweilige Gesprächskontext bleibt getrennt.


In [ ]:
GEWAEHLTES_FELD = '1.1'  # HIER ÄNDERN

if GEWAEHLTES_FELD not in chat_session.sections:
    raise KeyError(f'Unbekanntes Lebensfeld: {GEWAEHLTES_FELD}')
chat_session = chat_engine.select_section(chat_session.session_id, GEWAEHLTES_FELD)

print(f'Aktiv: {GEWAEHLTES_FELD} – {chat_session.sections[GEWAEHLTES_FELD].title}')
zeige_status(chat_session)
write_walkthrough_log(chat_engine, chat_session, PROTOKOLL_ORDNER)
print('Protokoll aktualisiert.')


## 10 – Lebensfeld freigeben

**Wann?** Erst nachdem du Ziel, Beobachtungen und Entwurf fachlich geprüft hast.

Die Freigabe kann einen fehlenden Entwurf erzeugen, markiert das Lebensfeld als bestätigt und wechselt zum nächsten aktiven Feld. Sie ist deshalb durch `FELD_FREIGEBEN = False` vor versehentlichem Ausführen geschützt.


In [ ]:
FELD_FREIGEBEN = False  # HIER ÄNDERN: Erst nach fachlicher Prüfung auf True setzen.

if FELD_FREIGEBEN:
    feld_id = chat_session.active_section_id
    chat_session = await chat_engine.confirm_section(chat_session.session_id, feld_id)
    print(f'Lebensfeld {feld_id} wurde freigegeben.')
    zeige_nachrichten(chat_session.messages[-2:])
    zeige_status(chat_session)
    write_walkthrough_log(chat_engine, chat_session, PROTOKOLL_ORDNER)
    print('Protokoll aktualisiert.')
else:
    print('Keine Freigabe ausgeführt. Setze FELD_FREIGEBEN bewusst auf True.')


## 11 – Vollständigen Gesprächsverlauf und Protokoll anzeigen

**Wann?** Bei Verständnisproblemen, vor der Abschlussbewertung oder wenn eine spätere Codex-Session den Test nachvollziehen soll.

Nach jedem Dialogschritt werden automatisch zwei stabile Dateien aktualisiert:

- `output/walkthrough/app_walkthrough_latest.md`: sichtbarer Gesprächsverlauf und Entwurfsstand.
- `output/walkthrough/app_walkthrough_latest_state.json`: Phase, aktives Lebensfeld und intern gespeicherte Beobachtungen, Unsicherheiten, Ziele und Entwürfe.

Die folgende Zelle zeigt zusätzlich den Verlauf direkt im Notebook und nennt die vollständigen Dateipfade. Eine neue Sitzung in Schritt 5 überschreibt die `latest`-Dateien bewusst mit dem neuen Testlauf.


In [ ]:
for nummer, nachricht in enumerate(chat_session.messages, start=1):
    rolle = 'Fachkraft' if nachricht.role.value == 'user' else 'Chatbot'
    feld = f' | Feld {nachricht.section_id}' if nachricht.section_id else ''
    print(f'\n{nummer}. {rolle}{feld}')
    print(nachricht.content)

protokoll_pfad, status_pfad = write_walkthrough_log(chat_engine, chat_session, PROTOKOLL_ORDNER)
print(f'\nDauerhaftes Chatprotokoll: {protokoll_pfad}')
print(f'Strukturierter Zustand: {status_pfad}')


## 12 – Bericht exportieren

**Wann?** Nach Prüfung der Entwürfe. Der Export beendet die Sitzung nicht. Du kannst danach weiter testen.

- Markdown eignet sich zur schnellen Inhaltsprüfung.
- DOCX prüft zusätzlich die tatsächliche Berichtsausgabe.

Die Dateien werden ausschließlich unter `output/walkthrough/` abgelegt. Der Schutzschalter verhindert einen unbeabsichtigten Export.


In [ ]:
EXPORTIEREN = False  # HIER ÄNDERN: Für den Testexport auf True setzen.

if EXPORTIEREN:
    ausgabeordner = REPO_ROOT / 'output/walkthrough'
    ausgabeordner.mkdir(parents=True, exist_ok=True)

    markdown_text, markdown_name = chat_engine.export_markdown(chat_session.session_id)
    markdown_pfad = ausgabeordner / markdown_name
    markdown_pfad.write_text(markdown_text, encoding='utf-8')

    docx_pfad = ausgabeordner / 'Max_Mustermann_Notebook_Test.docx'
    chat_engine.export_docx(chat_session.session_id, docx_pfad)

    print(f'Markdown: {markdown_pfad}')
    print(f'DOCX: {docx_pfad}')
else:
    print('Kein Export ausgeführt. Setze EXPORTIEREN bewusst auf True.')


## 13 – Fachliche Abschlussbewertung

Halte für einen auffälligen Dialog immer diese fünf Informationen fest:

1. Phase und aktives Lebensfeld.
2. Deine genaue Eingabe.
3. Vollständige Antwort des Chatbots.
4. Erwartetes fachliches Verhalten.
5. Abweichung im sichtbaren Dialog oder internen Zustand.

Ein guter Test prüft nicht nur freundliche Standardantworten. Verwende auch unvollständige Angaben, natürliche Umgangssprache, Rückfragen, Widerspruch, Korrekturen und Änderungswünsche. Personenbezogene Echtdaten gehören nicht in dieses Notebook.

### Was dieses Notebook nicht prüft

Das Notebook umgeht die Browseroberfläche. Es prüft deshalb keine Upload-Schaltflächen, Layouts, Ladeanzeigen, HTMX-Aktualisierungen oder Downloads im Browser. Diese Punkte prüfst du anschließend einmal direkt in der laufenden App.
